In [0]:
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")

In [0]:
catalog = dbutils.widgets.get("catalog_name")
schema_prefix = dbutils.widgets.get("schema_prefix")
silver_db = f"{catalog}.{schema_prefix}_silver"
gold_db = f"{catalog}.{schema_prefix}_gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {gold_db}")
print("Silver:", silver_db, "| Gold:", gold_db)

Silver: workspace.retail_silver | Gold: workspace.retail_gold


In [0]:
%run ./00b_pipeline_utils

Pipeline utils initializing. RUN_ID = 3a80d25c-2ed1-48be-a27c-0bfe5e2973f7 | ops schema = workspace.retail_ops


log_event() and quarantine_records() are ready to use.


[SUCCESS] (test) manual_check: First test of the logging system


run_id,layer,task_name,status,message,rows_affected,log_timestamp
3a80d25c-2ed1-48be-a27c-0bfe5e2973f7,test,manual_check,SUCCESS,First test of the logging system,0,2026-08-08T08:23:22.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_geolocation,SUCCESS,No rows failed (missing geolocation_zip_code_prefix),0,2026-08-08T08:11:00.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_reviews,SUCCESS,No rows failed (missing review_id or order_id),0,2026-08-08T08:10:53.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_payments,SUCCESS,No rows failed (negative payment_value),0,2026-08-08T08:10:46.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_payments,SUCCESS,No rows failed (missing order_id/payment_sequential),0,2026-08-08T08:10:44.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_payments,SUCCESS,No rows failed (negative payment_value),0,2026-08-08T08:03:18.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_payments,SUCCESS,No rows failed (missing order_id/payment_sequential),0,2026-08-08T08:03:15.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_items,SUCCESS,No rows failed (negative freight_value),0,2026-08-08T08:01:26.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_items,SUCCESS,No rows failed (negative price),0,2026-08-08T08:01:24.000Z
7ccdb0d8-0864-4803-b643-b4582805be16,quarantine,silver_order_items,SUCCESS,No rows failed (missing order_id/order_item_id),0,2026-08-08T08:01:22.000Z


In [0]:
from pyspark.sql.functions import (
    col, sha2, concat_ws, date_format, datediff, expr, lit,
    coalesce, when, avg, dayofweek
)

In [0]:
date_df = (spark.range(0, 365 * 4 + 1)
           .select(expr("date_add(cast('2016-01-01' as date), cast(id as int))").alias("date")))

dim_date = (date_df
            .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("integer"))
            .withColumn("year", date_format(col("date"), "yyyy").cast("integer"))
            .withColumn("month", date_format(col("date"), "MM").cast("integer"))
            .withColumn("month_name", date_format(col("date"), "MMMM"))
            .withColumn("day", date_format(col("date"), "dd").cast("integer"))
            .withColumn("day_of_week", dayofweek(col("date")))
            .withColumn("day_name", date_format(col("date"), "EEEE"))
            .withColumn("quarter", date_format(col("date"), "q").cast("integer"))
            .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), 1).otherwise(0)))

dim_date.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_db}.DimDate")
log_event("gold", "DimDate", "SUCCESS", rows_affected=dim_date.count())

[SUCCESS] (gold) DimDate: 


In [0]:
dim_location = (spark.table(f"{silver_db}.geolocation")
                 .withColumn("location_key", sha2(col("geolocation_zip_code_prefix"), 256))
                 .select(
                     col("location_key"),
                     col("geolocation_zip_code_prefix").alias("zip_code_prefix"),
                     col("geolocation_city").alias("city"),
                     col("geolocation_state").alias("state"),
                     col("geolocation_lat").alias("latitude"),
                     col("geolocation_lng").alias("longitude")
                 ))

dim_location.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_db}.DimLocation")
log_event("gold", "DimLocation", "SUCCESS", rows_affected=dim_location.count())

[SUCCESS] (gold) DimLocation: 


In [0]:
dim_customer = (spark.table(f"{silver_db}.customers")
                 .withColumn("customer_key", sha2(col("customer_unique_id"), 256))
                 .withColumn("location_key", sha2(col("customer_zip_code_prefix"), 256))
                 .select(
                     col("customer_key"), col("customer_unique_id"), col("customer_id"),
                     col("customer_zip_code_prefix").alias("zip_code_prefix"),
                     col("customer_city").alias("city"), col("customer_state").alias("state"),
                     col("location_key")
                 ))

dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_db}.DimCustomer")
log_event("gold", "DimCustomer", "SUCCESS", rows_affected=dim_customer.count())

[SUCCESS] (gold) DimCustomer: 


In [0]:
dim_product = (spark.table(f"{silver_db}.products")
               .join(spark.table(f"{silver_db}.product_category_name_translation"), "product_category_name", "left")
               .withColumn("product_key", sha2(col("product_id"), 256))
               .select(
                   col("product_key"), col("product_id"),
                   col("product_category_name").alias("category_name"),
                   coalesce(col("product_category_name_english"), col("product_category_name")).alias("category_name_english"),
                   col("product_weight_g").alias("weight_g"),
                   col("product_length_cm").alias("length_cm"),
                   col("product_height_cm").alias("height_cm"),
                   col("product_width_cm").alias("width_cm")
               ))

dim_product.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_db}.DimProduct")
log_event("gold", "DimProduct", "SUCCESS", rows_affected=dim_product.count())

[SUCCESS] (gold) DimProduct: 


In [0]:
dim_seller = (spark.table(f"{silver_db}.sellers")
              .withColumn("seller_key", sha2(col("seller_id"), 256))
              .withColumn("location_key", sha2(col("seller_zip_code_prefix"), 256))
              .select(
                  col("seller_key"), col("seller_id"),
                  col("seller_zip_code_prefix").alias("zip_code_prefix"),
                  col("seller_city").alias("city"), col("seller_state").alias("state"),
                  col("location_key")
              ))

dim_seller.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_db}.DimSeller")
log_event("gold", "DimSeller", "SUCCESS", rows_affected=dim_seller.count())

[SUCCESS] (gold) DimSeller: 


In [0]:
orders = spark.table(f"{silver_db}.orders")
items = spark.table(f"{silver_db}.order_items")
payments = spark.table(f"{silver_db}.order_payments")
reviews = spark.table(f"{silver_db}.order_reviews")
customers = spark.table(f"{gold_db}.DimCustomer")

order_totals = items.groupBy("order_id").sum("price").withColumnRenamed("sum(price)", "total_price")
order_payments = payments.groupBy("order_id").sum("payment_value").withColumnRenamed("sum(payment_value)", "total_payment")
items_with_totals = items.join(order_totals, "order_id", "inner")
avg_reviews = reviews.groupBy("order_id").agg(avg("review_score").alias("review_score"))

fact_sales = (items_with_totals
    .join(orders, "order_id", "inner")
    .join(order_payments, "order_id", "left")
    .join(avg_reviews, "order_id", "left")
    .join(customers, "customer_id", "inner")
    .withColumn("sales_key", sha2(concat_ws("-", col("order_id"), col("order_item_id")), 256))
    .withColumn("product_key", sha2(col("product_id"), 256))
    .withColumn("seller_key", sha2(col("seller_id"), 256))
    .withColumn("location_key", sha2(col("zip_code_prefix"), 256))
    .withColumn("order_purchase_date_key", date_format(col("order_purchase_timestamp"), "yyyyMMdd").cast("integer"))
    .withColumn("prorated_payment_value",
        when(col("total_price") > 0,
             (col("price") / col("total_price")) * coalesce(col("total_payment"), col("price") + col("freight_value")))
        .otherwise(col("price") + col("freight_value")))
    .withColumn("actual_delivery_time_days", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp")))
    .withColumn("estimated_delivery_time_days", datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp")))
    .withColumn("delivery_delay_days",
        when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
             datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date")))
        .otherwise(lit(0)))
    .select(
        "sales_key", "order_id", "order_item_id", "customer_key", "product_key", "seller_key", "location_key",
        "order_purchase_date_key", "order_status", "order_purchase_timestamp", "price", "freight_value",
        col("prorated_payment_value").alias("payment_value"), "review_score",
        "actual_delivery_time_days", "estimated_delivery_time_days", "delivery_delay_days"
    ))

row_count = merge_upsert(fact_sales, f"{gold_db}.FactSales", ["sales_key"])
log_event("gold", "FactSales", "SUCCESS", rows_affected=row_count)

workspace.retail_gold.FactSales: merged, now 112650 rows total
[SUCCESS] (gold) FactSales: 


In [0]:
display(spark.sql(f"SHOW TABLES IN {gold_db}"))
total_rev = spark.table(f"{gold_db}.FactSales").select(expr("sum(price)")).collect()[0][0]
print(f"FactSales rows: {spark.table(f'{gold_db}.FactSales').count()}")
print(f"Total Revenue: R$ {total_rev:,.2f}")

database,tableName,isTemporary
retail_gold,data_quality_logs,false
retail_gold,dimcustomer,false
retail_gold,dimdate,false
retail_gold,dimlocation,false
retail_gold,dimproduct,false
retail_gold,dimseller,false
retail_gold,factsales,false


FactSales rows: 112650
Total Revenue: R$ 13,591,643.70
